# Lesson 16 — chunking · หั่นข้อความยาวก่อนเก็บ

โพสต์ยาวหนึ่งโพสต์ embed เป็น vector เดียว ความหมายเฉลี่ยกันจนจาง
หั่นเป็นชิ้นเล็ก แต่ละชิ้นมี vector ของตัวเอง ค้นเจอชิ้น แล้วค่อยเอาโพสต์แม่มาแสดง
บทนี้ไม่มี API ใหม่ของ LanceDB เลย เป็นเรื่องออกแบบ schema ล้วน ๆ

ภาษาไทยไม่มีช่องว่างระหว่างคำ หั่นตามจำนวนตัวอักษรคือวิธีตรงไปตรงมาที่สุด
40 ตัวอักษร ทับซ้อน 10 ตัว เพื่อไม่ให้คำขาดกลางแล้วหายไปทั้งสองฝั่ง
ตัดตามคำด้วย icu (บทที่ 8) ดีกว่า แต่บทนี้ขอแบบเห็นตัวเลขชัด ๆ ก่อน

In [1]:
%pip install -q lancedb pandas duckdb

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, urllib.request, pathlib
if not pathlib.Path("../data/lesson_data.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Soul-Brews-Studio/lancedb-oracle/main/lessons/data/lesson_data.py", "lesson_data.py")
sys.path.insert(0, "../data")
from lesson_data import load

import pandas as pd

posts = load("nat_posts.jsonl")
longest = sorted(posts, key=lambda p: -len(p["text"]))[:3]
pd.DataFrame([{"id": p["id"], "topic": p["topic"], "chars": len(p["text"]), "text": p["text"][:40]} for p in longest])

,id,topic,chars,text
0,p01,memory,184,ยอมกลับมาทำ Memory เพราะซาบซึ้งว่ามันต้อ
1,p08,agents,177,Multi Agent แบบ Team .... ของ Claude Cod
2,p07,agents,159,เข้าสู่ยุค Multi-Agents แบบเต็มตัว.... C


**หั่น** — เลื่อนหน้าต่าง 40 ตัวอักษร ก้าวทีละ 30 (40 − 10 ทับซ้อน)
ตารางข้างล่างคือโพสต์ยาวสุด (p01, 184 ตัวอักษร) หั่นได้ 6 ชิ้น
ดู column `overlap` = 10 ตัวอักษรท้ายของชิ้นก่อน ต้องเท่ากับ 10 ตัวอักษรแรกของชิ้นนี้ทุกแถว
ชิ้นสุดท้ายสั้นกว่า 40 เพราะข้อความหมดก่อน

In [3]:
SIZE, OVERLAP = 40, 10

def chunk(text, size=SIZE, overlap=OVERLAP):
    step = size - overlap
    return [text[i:i + size] for i in range(0, max(len(text) - overlap, 1), step)]

p = longest[0]
pieces = chunk(p["text"])
pd.DataFrame([{
    "chunk_index": i,
    "start": i * (SIZE - OVERLAP),
    "end": i * (SIZE - OVERLAP) + len(c),
    "chars": len(c),
    "overlap (prev tail = this head)": f"{pieces[i-1][-OVERLAP:]} = {c[:OVERLAP]}" if i else "",
    "text": c,
} for i, c in enumerate(pieces)])

,chunk_index,start,end,chars,overlap (prev tail = this head),text
0,0,0,40,40,,ยอมกลับมาทำ Memory เพราะซาบซึ้งว่ามันต้อ
1,1,30,70,40,งว่ามันต้อ = งว่ามันต้อ,งว่ามันต้องทำแล้วครับ ARRA Oracle Memory
2,2,60,100,40,cle Memory = cle Memory,cle Memory สามารถทำให้ claude บนเว็ป คุย
3,3,90,130,40,บนเว็ป คุย = บนเว็ป คุย,บนเว็ป คุยกับชาวแก๊งอื่นๆได้แล้วครับ แล้
4,4,120,160,40,้วครับ แล้ = ้วครับ แล้,้วครับ แล้วจะพาทำ Workshop สร้าง Memory
5,5,150,184,34,าง Memory = าง Memory,าง Memory และ MCP ด้วยตัวเองนะครับ


**เก็บลงตาราง** — vector ของแต่ละชิ้นทำมือเหมือนบทก่อน ๆ
ชิ้นของโพสต์ไหน ก็ยืม vector ของโพสต์นั้น แล้วขยับนิดหน่อยตามลำดับชิ้น
(ของจริงใช้โมเดล embed ทีละชิ้น บทที่ 7)
สามโพสต์ได้ 17 ชิ้น ตาราง `chunks` มี `parent_id` ชี้กลับ `posts`

In [4]:
import lancedb

rows = []
for p in longest:
    for i, c in enumerate(chunk(p["text"])):
        v = [round(x + 0.02 * i, 3) for x in p["vector"]]
        rows.append({"chunk_id": f"{p['id']}#{i}", "parent_id": p["id"], "chunk_index": i, "text": c, "vector": v})

db = lancedb.connect("./data")
chunks = db.create_table("chunks", data=rows, mode="overwrite")
parents = db.create_table("posts", data=longest, mode="overwrite")
summary = pd.DataFrame([{"parent_id": p["id"], "chars": len(p["text"]), "chunks": len(chunk(p["text"]))} for p in longest])
summary.loc[len(summary)] = ["total", summary.chars.sum(), summary.chunks.sum()]
summary

[2026-09-10T11:49:45Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/16-chunking/data/chunks.lance, it will be created
[2026-09-10T11:49:45Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/16-chunking/data/posts.lance, it will be created


,parent_id,chars,chunks
0,p01,184,6
1,p08,177,6
2,p07,159,5
3,total,520,17


**ค้นชิ้น** — คำถามทิศ memory `[1, 0, 0]` ขอ 8 ชิ้น
6 ชิ้นแรกมาจาก p01 ทั้งหมด (`_distance` 0.02 ถึง 0.14) ถ้าส่งแบบนี้ให้ agent มันจะเห็นโพสต์เดิมซ้ำ 6 ครั้ง
ชิ้นที่ 7–8 ค่อยเป็น p08

In [5]:
hits = chunks.search([1.0, 0.0, 0.0]).limit(8).to_pandas()
hits[["chunk_id", "parent_id", "chunk_index", "_distance", "text"]].assign(text=lambda d: d.text.str[:30])

,chunk_id,parent_id,chunk_index,_distance,text
0,p01#0,p01,0,0.0200,ยอมกลับมาทำ Memory เพราะซาบซึ้
1,p01#1,p01,1,0.0212,งว่ามันต้องทำแล้วครับ ARRA Ora
2,p01#2,p01,2,0.0248,cle Memory สามารถทำให้ claude
3,p01#3,p01,3,0.0308,บนเว็ป คุยกับชาวแก๊งอื่นๆได้แล
4,p01#4,p01,4,0.0392,้วครับ แล้วจะพาทำ Workshop สร้
5,p01#5,p01,5,0.0500,าง Memory และ MCP ด้วยตัวเองนะ
6,p08#0,p08,0,1.6200,Multi Agent แบบ Team .... ของ
7,p08#1,p08,1,1.6212,Claude Code เพื่อเรียก SendMes


**ยุบกลับเป็นโพสต์แม่** — group ด้วย `parent_id` เอาชิ้นที่ใกล้ที่สุดเป็นคะแนนของโพสต์
แล้ว join กลับไปเอาข้อความเต็ม
8 ชิ้นกลายเป็น 2 โพสต์ p01 (0.02) กับ p08 (1.62) agent เห็นแต่ละโพสต์ครั้งเดียว พร้อมชิ้นที่ทำให้เจอ
นี่คือ pattern "chunk to retrieve, parent to show" ที่ระบบ RAG ทุกตัวใช้

In [6]:
import duckdb
best = hits.groupby("parent_id", as_index=False).agg(best_distance=("_distance", "min"), best_chunk=("chunk_id", "first"), chunks_hit=("chunk_id", "count"))
parent_df = parents.to_pandas()[["id", "topic", "text"]]
duckdb.sql("""
    SELECT b.parent_id, p.topic, b.best_distance, b.chunks_hit, b.best_chunk, substr(p.text, 1, 45) AS parent_text
    FROM best b JOIN parent_df p ON p.id = b.parent_id
    ORDER BY b.best_distance
""").df()

,parent_id,topic,best_distance,chunks_hit,best_chunk,parent_text
0,p01,memory,0.02,6,p01#0,ยอมกลับมาทำ Memory เพราะซาบซึ้งว่ามันต้องทำแล
1,p08,agents,1.62,2,p08#0,Multi Agent แบบ Team .... ของ Claude Code เพื


**ราคาของทับซ้อน** — ตัวอักษรรวมของชิ้นมากกว่าโพสต์ต้นทาง
ทับซ้อน 10/40 = จ่ายเพิ่มราว 25% (วัดจริง 27% เพราะชิ้นท้ายของแต่ละโพสต์สั้นไม่เต็ม 40) เพื่อไม่ให้คำขาด
บน disk `chunks` ใหญ่กว่า `posts` ตามนั้น

In [7]:
from pathlib import Path
chunk_chars = sum(len(r["text"]) for r in rows)
post_chars = sum(len(p["text"]) for p in longest)
pd.DataFrame([
    {"table": "posts", "rows": len(longest), "chars": post_chars, "overhead": "",
     "bytes on disk": sum(f.stat().st_size for f in Path("data/posts.lance/data").glob("*.lance"))},
    {"table": "chunks", "rows": len(rows), "chars": chunk_chars, "overhead": f"+{100 * (chunk_chars / post_chars - 1):.0f}%",
     "bytes on disk": sum(f.stat().st_size for f in Path("data/chunks.lance/data").glob("*.lance"))},
])

,table,rows,chars,overhead,bytes on disk
0,posts,3,520,,2628
1,chunks,17,660,+27%,3602
